# 06 · How CPython Runs Your Code: Tokens, AST, Bytecode, the Interpreter & JIT

**Module 10 – Advanced Python: Internals & Metaprogramming** · Estimated time: 50 min

### 🎯 What you will learn
- The pipeline: source → tokens → **AST** → **bytecode** → evaluation loop
- Reading and transforming code with `tokenize` and `ast`
- Code objects and **`dis`**: reading bytecode
- `.pyc` files and `__pycache__`; `-O`, `-X importtime`
- The **specializing adaptive interpreter** (3.11+) and the **JIT** (3.13+, much improved in 3.15)
- Frames, the GIL in one paragraph, and other Python implementations (PyPy, GraalPy, MicroPython)

## 1. The pipeline

```
  source code (.py)
       │  tokenizer
       ▼
  tokens (NAME, OP, NUMBER, ...)
       │  parser (PEG)
       ▼
  AST (abstract syntax tree)
       │  compiler (+ optimisations: constant folding)
       ▼
  code object (bytecode + constants + names)   ──► cached as __pycache__/*.pyc
       │
       ▼
  evaluation loop (ceval) ── specialised instructions ── JIT (optional) ──► results
```

## 2. Tokens

In [1]:
import tokenize, io

src = "total = price * 1.18  # with tax\n"
for tok in tokenize.generate_tokens(io.StringIO(src).readline):
    print(f"{tokenize.tok_name[tok.type]:<10} {tok.string!r}")

NAME       'total'
OP         '='
NAME       'price'
OP         '*'
NUMBER     '1.18'
COMMENT    '# with tax'
NEWLINE    '\n'
ENDMARKER  ''


## 3. The AST

In [2]:
import ast

tree = ast.parse("total = price * 1.18 if price > 0 else 0")
print(ast.dump(tree, indent=2))

Module(
  body=[
    Assign(
      targets=[
        Name(id='total', ctx=Store())],
      value=IfExp(
        test=Compare(
          left=Name(id='price'),
          ops=[
            Gt()],
          comparators=[
            Constant(value=0)]),
        body=BinOp(
          left=Name(id='price'),
          op=Mult(),
          right=Constant(value=1.18)),
        orelse=Constant(value=0)))])


### Analysing code with the AST (how linters work)

In [3]:
code = '''
import os, json
def load(path):
    print("loading", path)
    data = json.load(open(path))
    eval(data["expr"])
    return data
'''
class Auditor(ast.NodeVisitor):
    def visit_Call(self, node):
        if isinstance(node.func, ast.Name) and node.func.id in {"eval", "exec", "print"}:
            print(f"line {node.lineno}: call to {node.func.id}()")
        self.generic_visit(node)
    def visit_Import(self, node):
        print(f"line {node.lineno}: imports {[a.name for a in node.names]}")

Auditor().visit(ast.parse(code))

line 2: imports ['os', 'json']
line 4: call to print()
line 6: call to eval()


### Transforming code (how tools like pytest rewrite `assert`)

In [4]:
class SwapAddToMul(ast.NodeTransformer):
    def visit_BinOp(self, node):
        self.generic_visit(node)
        if isinstance(node.op, ast.Add):
            return ast.copy_location(ast.BinOp(node.left, ast.Mult(), node.right), node)
        return node

tree = SwapAddToMul().visit(ast.parse("result = 6 + 7"))
ast.fix_missing_locations(tree)
print(ast.unparse(tree))
ns = {}
exec(compile(tree, "<transformed>", "exec"), ns)
print(ns["result"])

result = 6 * 7
42


## 4. Code objects and bytecode

In [5]:
import dis

def price_with_tax(price, rate=0.18):
    if price < 0:
        raise ValueError("negative")
    return round(price * (1 + rate), 2)

co = price_with_tax.__code__
print("args     :", co.co_varnames[:co.co_argcount])
print("locals   :", co.co_varnames)
print("constants:", co.co_consts)
print("names    :", co.co_names)
print("first line:", co.co_firstlineno, "| bytes of bytecode:", len(co.co_code))
dis.dis(price_with_tax)

args     : ('price', 'rate')
locals   : ('price', 'rate')
constants: (0, 'negative')
names    : ('ValueError', 'round')
first line: 3 | bytes of bytecode: 92
  3           RESUME                   0

  4           LOAD_FAST_BORROW         0 (price)
              LOAD_SMALL_INT           0
              COMPARE_OP              18 (bool(<))
              POP_JUMP_IF_FALSE       12 (to L1)
              NOT_TAKEN

  5           LOAD_GLOBAL              1 (ValueError + NULL)
              LOAD_CONST               1 ('negative')
              CALL                     1
              RAISE_VARARGS            1

  6   L1:     LOAD_GLOBAL              3 (round + NULL)
              LOAD_FAST_BORROW         0 (price)
              LOAD_SMALL_INT           1
              LOAD_FAST_BORROW         1 (rate)
              BINARY_OP                0 (+)
              BINARY_OP                5 (*)
              LOAD_SMALL_INT           2
              CALL                     2
              RETURN_

Bytecode is a stack machine: `LOAD_FAST` (or its cheaper 3.14+ variant `LOAD_FAST_BORROW`) pushes a local, `LOAD_SMALL_INT`/`LOAD_CONST` push constants, `BINARY_OP` pops two values and pushes the result, `RETURN_VALUE` returns the top of the stack.

> 💡 Bytecode is an **internal detail that changes in every Python version**: opcodes are added, renamed and fused. Never depend on it in production code; use it for understanding and debugging.

### Compile-time optimisations: constant folding

In [6]:
def seconds_per_week():
    return 60 * 60 * 24 * 7          # computed once by the compiler

dis.dis(seconds_per_week)

  1           RESUME                   0

  2           LOAD_CONST               1 (604800)
              RETURN_VALUE


### Why local variables are faster than globals

In [7]:
COUNT = 10
def use_global():
    return COUNT + 1
def use_local(count=10):
    return count + 1

print([i.opname for i in dis.get_instructions(use_global) if "LOAD" in i.opname])
print([i.opname for i in dis.get_instructions(use_local) if "LOAD" in i.opname])

['LOAD_GLOBAL', 'LOAD_SMALL_INT']
['LOAD_FAST_BORROW', 'LOAD_SMALL_INT']


`LOAD_FAST*` reads an array slot; `LOAD_GLOBAL` needs dictionary lookups (globals, then builtins), although specialisation (section 6) makes it much cheaper than it used to be.

## 5. `.pyc` files and `__pycache__`

When you **import** a module, CPython caches its bytecode in `__pycache__/name.cpython-315.pyc` and reuses it while the source is unchanged (checked by timestamp or hash).

In [8]:
import pathlib, py_compile, importlib.util
pathlib.Path("tiny_mod.py").write_text("def f():\n    return 42\n")
cached = py_compile.compile("tiny_mod.py")
print(cached)
print(importlib.util.cache_from_source("tiny_mod.py"))

__pycache__/tiny_mod.cpython-315.pyc
__pycache__/tiny_mod.cpython-315.pyc


| Option | Effect |
|---|---|
| `python -O` | removes `assert` statements, sets `__debug__ = False` (`.opt-1.pyc`) |
| `python -OO` | also removes docstrings |
| `python -B` / `PYTHONDONTWRITEBYTECODE=1` | don't write `.pyc` files |
| `python -m compileall src/` | pre-compile a tree (Docker images, read-only installs) |
| `python -X importtime app.py` | how long each import takes (find slow startup) |

In [9]:
import subprocess, sys
out = subprocess.run([sys.executable, "-X", "importtime", "-c", "import json"], capture_output=True, text=True).stderr
print("\n".join(out.splitlines()[-6:]))

import time:       293 |       3396 |     re
import time:        28 |         28 |       _json
import time:       228 |        256 |     json.scanner
import time:       267 |       3917 |   json.decoder
import time:       200 |        200 |   json.encoder
import time:       165 |       4281 | json


## 6. The specializing adaptive interpreter (3.11+)

Since Python 3.11, the interpreter **watches your code run** and replaces generic instructions with **specialised** ones for the types it actually sees (e.g. `BINARY_OP_ADD_INT`, `LOAD_ATTR_INSTANCE_VALUE`). This is a big part of why 3.11+ is 10–60% faster.

In [10]:
def add_numbers(a, b):
    return a + b

for _ in range(1000):                       # warm up with ints
    add_numbers(1, 2)
dis.dis(add_numbers, adaptive=True)         # shows the specialised instructions

  1           RESUME_CHECK             0

  2           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (a, b)
              BINARY_OP_ADD_INT        0 (+)
              RETURN_VALUE


## 7. The JIT compiler (3.13+)

- 3.13 added an experimental **copy-and-patch JIT** (built with `--enable-experimental-jit`), which turns hot traces of specialised bytecode into machine code
- 🆕 **3.15** significantly upgraded it: the release notes report ~7–8% faster on x86-64 Linux and ~11–12% on AArch64 macOS where it's enabled
- Toggle in JIT-enabled builds with `PYTHON_JIT=1` / `PYTHON_JIT=0`; check with `sys._jit.is_enabled()` where available

In [11]:
jit = getattr(sys, "_jit", None)
if jit is not None:
    print("JIT available:", jit.is_available(), "| enabled:", jit.is_enabled())
else:
    print("This build has no sys._jit interface")

JIT available: True | enabled: False


## 8. Frames and the GIL in brief

- Each function call gets a **frame**: its locals, the instruction pointer and a link to the caller (Module 10 · 04 showed `inspect.currentframe()`). In 3.11+ frames are lightweight and created lazily.
- The **GIL** (Global Interpreter Lock) lets only one thread run Python bytecode at a time in standard builds; the free-threaded build (`python3.15t`) removes it (Module 13 · 08).

## 9. Python implementations

| Implementation | Notes |
|---|---|
| **CPython** | the reference implementation (C), what you get from python.org |
| **PyPy** | tracing JIT; often 4–10× faster for pure-Python loops; C-extension support via a compatibility layer |
| **GraalPy** | on GraalVM; Java interop |
| **MicroPython / CircuitPython** | microcontrollers |
| **Pyodide** | CPython compiled to WebAssembly (runs in browsers) |
| **Cython / mypyc / Nuitka** | compile Python(-like) code to C extensions or executables (lesson 09) |

In [12]:
import platform
print(platform.python_implementation(), platform.python_version(), platform.python_compiler())

CPython 3.15.0 Clang 23.1.3 


## 📝 Summary
- Source → tokens → AST → bytecode (code objects) → the evaluation loop; bytecode is cached in `__pycache__`.
- `ast` lets you analyse and transform code (linters, pytest); `dis` shows bytecode.
- Locals are faster than globals; constants are folded at compile time.
- 3.11+ specialises bytecode adaptively; 3.13+ has a JIT, much improved in 3.15.

## 🏋️ Practice
1. Use `ast` to list every function name and its number of arguments in a Python file.
2. Write a `NodeVisitor` that warns about bare `except:` clauses.
3. Compare `dis.dis` output for `[x * 2 for x in data]` and `list(map(lambda x: x * 2, data))`.

<details><summary>Show solutions for 1–2</summary>

```python
import ast

source = '''
def a(x, y): pass
def b():
    try:
        pass
    except:
        pass
'''
tree = ast.parse(source)
for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        print(node.name, len(node.args.args))

class BareExcept(ast.NodeVisitor):
    def visit_ExceptHandler(self, node):
        if node.type is None:
            print(f"line {node.lineno}: bare except")
        self.generic_visit(node)

BareExcept().visit(tree)
```
</details>